#### Compare Each Model using NDGC@K

In [ ]:
from itertools import product

import numpy as np
import pandas as pd

from surprise import KNNBasic
from surprise.model_selection import train_test_split
from sklearn.metrics import ndcg_score


# Same split for every parameter combination
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)


def calculate_ndcg(predictions, k=10):
    """
    Calculate average NDCG@k across users.

    Uses actual ratings as graded relevance.
    """

    rows = []

    for pred in predictions:
        rows.append({
            "user": pred.uid,
            "actual": pred.r_ui,
            "predicted": pred.est,
        })

    pred_df = pd.DataFrame(rows)

    user_scores = []

    for user_id, group in pred_df.groupby("user"):

        # Need at least 2 items to meaningfully rank
        if len(group) < 2:
            continue

        true_relevance = group["actual"].to_numpy()
        predicted_scores = group["predicted"].to_numpy()

        score = ndcg_score(
            [true_relevance],
            [predicted_scores],
            k=min(k, len(group)),
        )

        user_scores.append(score)

    return np.mean(user_scores)